# Users handling

Notebook generado a partir de `users_handling.py`.

In [2]:
import os
import tarfile

import pandas as pd

In [3]:
tar_file = "oasis_cross-sectional_disc1.tar.gz"
xlsx_file = "oasis_cross-sectional-5708aa0a98d82080.xlsx"
output_dir = "result_tar2.tar.gz"

In [6]:
df = pd.read_excel(xlsx_file)

# Delete unused columns
df.drop(columns=["M/F", "Hand", "Educ", "SES", "eTIV", "ASF", "Delay"], inplace=True)

df.head()

,ID,Age,MMSE,CDR,nWBV
0,OAS1_0001_MR1,74,29.0,0.0,0.743
1,OAS1_0002_MR1,55,29.0,0.0,0.810
2,OAS1_0003_MR1,73,27.0,0.5,0.708
3,OAS1_0004_MR1,28,NaN,NaN,0.803
4,OAS1_0005_MR1,18,NaN,NaN,0.848


In [7]:
# Create set of IDs to exclude
excluded_ids = set()

# Exclude patients without MMSE and CDR data
no_data = df[(df["MMSE"].isnull()) & (df["CDR"].isnull())]["ID"]
excluded_ids.update(no_data.to_list())

# Exclude MR2 records
mr2_records = df[df["ID"].str.contains("MR2", na=False)]["ID"]
excluded_ids.update(mr2_records.to_list())

for ele in excluded_ids:
    print(f"Excluding patient {ele}")

print(f"\nTotal excluded: {len(excluded_ids)}")

Excluding patient OAS1_0150_MR1
Excluding patient OAS1_0136_MR1
Excluding patient OAS1_0159_MR1
Excluding patient OAS1_0004_MR1
Excluding patient OAS1_0311_MR1
Excluding patient OAS1_0448_MR1
Excluding patient OAS1_0156_MR2
Excluding patient OAS1_0017_MR1
Excluding patient OAS1_0104_MR1
Excluding patient OAS1_0211_MR1
Excluding patient OAS1_0014_MR1
Excluding patient OAS1_0258_MR1
Excluding patient OAS1_0218_MR1
Excluding patient OAS1_0302_MR1
Excluding patient OAS1_0127_MR1
Excluding patient OAS1_0054_MR1
Excluding patient OAS1_0296_MR1
Excluding patient OAS1_0386_MR1
Excluding patient OAS1_0061_MR1
Excluding patient OAS1_0117_MR2
Excluding patient OAS1_0346_MR1
Excluding patient OAS1_0163_MR1
Excluding patient OAS1_0224_MR1
Excluding patient OAS1_0043_MR1
Excluding patient OAS1_0027_MR1
Excluding patient OAS1_0236_MR1
Excluding patient OAS1_0310_MR1
Excluding patient OAS1_0397_MR1
Excluding patient OAS1_0265_MR1
Excluding patient OAS1_0410_MR1
Excluding patient OAS1_0222_MR1
Excludin

Variables de interes:

- Age
- MMSE
- CDR
- nWBV

In [8]:
def filter_pacients(input_tar_file, output_tar_path, ids_to_exclude):
    with tarfile.open(input_tar_file, "r:gz") as tar_in:
        with tarfile.open(output_tar_path, "w:gz") as tar_out:
            members = tar_in.getmembers()

            for member in members:
                path_parts = member.name.split("/")

                if len(path_parts) >= 2:
                    patient_id = path_parts[1]
                    if patient_id not in ids_to_exclude:
                        extracted = tar_in.extractfile(member)
                        tar_out.addfile(member, extracted)
                else:
                    extracted = tar_in.extractfile(member)
                    tar_out.addfile(member, extracted)

In [9]:
filter_pacients(tar_file, output_dir, excluded_ids)
print(f"Archivo generado: {output_dir}")

Archivo generado: result_tar2.tar.gz
